# 1. Duplicate Data & Safe Type Casting

This notebook covers:
1. Identifying and resolving **Exact Duplicates** vs. **Subset (Semantic) Duplicates**.
2. Fixing incorrect data types and applying safe numeric/datetime conversions using `errors='coerce'`.
3. Practical rules to prevent duplicate-based **Data Leakage**.

In [27]:
import numpy as np
import pandas as pd

# Creating a messy raw dataset with duplicate records and broken data types
raw_data = {
    'Transaction_ID': [1001, 1002, 1003, 1001, 1004, 1002, 1005],
    'Customer_Name': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', 'Bob', 'Eve'],
    'Age': ['25', '30', 'unknown', '25', '45', '30', '29'],
    'Amount_Paid': ['$150.00', '$200.50', '$99.00', '$150.00', '$320.00', '$210.00', 'free'],
    'City': ['Hyderabad', 'Bengaluru', 'Delhi', 'Hyderabad', 'Mumbai', 'Bangalore', 'Chennai']
}

df = pd.DataFrame(raw_data)
print("=== RAW MESSY DATASET ===")
display(df)
print("\nData Types:")
print(df.dtypes)

=== RAW MESSY DATASET ===


,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25,$150.00,Hyderabad
1,1002,Bob,30,$200.50,Bengaluru
2,1003,Charlie,unknown,$99.00,Delhi
3,1001,Alice,25,$150.00,Hyderabad
4,1004,David,45,$320.00,Mumbai
5,1002,Bob,30,$210.00,Bangalore
6,1005,Eve,29,free,Chennai



Data Types:
Transaction_ID     int64
Customer_Name     object
Age               object
Amount_Paid       object
City              object
dtype: object


---
## Part 1: Duplicate Detection & Removal

- **Exact Duplicates:** Every single column value is identical across rows.
- **Subset / Semantic Duplicates:** A primary key or business ID (e.g., `Transaction_ID`) matches, but non-key columns may have slight updates.
- **Leakage Rule:** Deduplication must be performed **BEFORE** the train/test split.

In [28]:
df.duplicated()

0    False
1    False
2    False
3     True
4    False
5    False
6    False
dtype: bool

In [29]:
new_df=df.copy()
new_df = new_df.drop_duplicates(subset=['Transaction_ID'], keep='first').reset_index(drop=True)
print("\n=== DATASET AFTER DROPPING DUPLICATES BASED ON 'Transaction_ID' ===")
display(new_df)


=== DATASET AFTER DROPPING DUPLICATES BASED ON 'Transaction_ID' ===


,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25,$150.00,Hyderabad
1,1002,Bob,30,$200.50,Bengaluru
2,1003,Charlie,unknown,$99.00,Delhi
3,1004,David,45,$320.00,Mumbai
4,1005,Eve,29,free,Chennai


A new function: `.reset_index(drop=True)`: this will reset the index, after row deletion
> Note: this is not needed in real world, as the index in the dataframe comes from pandas not the real dataset

---
## Part 2: Safe Data Type Casting

Features frequently arrive stored as `object` (string) due to formatting symbols (like `$`) or text placeholders (like `'unknown'` or `'free'`).

### Best Practices:
1. Clean strings using `.str.replace()` before casting.
2. Use `pd.to_numeric(..., errors='coerce')` to turn corrupted entries into `NaN` safely without raising execution errors.

In [30]:
# 1. Clean 'Amount_Paid': remove '$' symbol and safely convert to float
new_df['Amount_Paid'] = new_df['Amount_Paid'].str.replace('$', '', regex=False)
new_df['Amount_Paid'] = pd.to_numeric(new_df['Amount_Paid'], errors='coerce')

# 2. Convert 'Age': 'unknown' strings will automatically become NaN
new_df['Age'] = pd.to_numeric(new_df['Age'], errors='coerce')

# 3. Optimize 'City' data type to category
new_df['City'] = new_df['City'].astype('category')

print("=== FINAL CLEANED & PROPERLY TYPED DATAFRAME ===")
display(new_df)
print("\nUpdated Column Types:")
print(new_df.dtypes)

=== FINAL CLEANED & PROPERLY TYPED DATAFRAME ===


,Transaction_ID,Customer_Name,Age,Amount_Paid,City
0,1001,Alice,25.0,150.0,Hyderabad
1,1002,Bob,30.0,200.5,Bengaluru
2,1003,Charlie,NaN,99.0,Delhi
3,1004,David,45.0,320.0,Mumbai
4,1005,Eve,29.0,NaN,Chennai



Updated Column Types:
Transaction_ID       int64
Customer_Name       object
Age                float64
Amount_Paid        float64
City              category
dtype: object


new functions: ` errors='coerce' `: this is commonly used with `df.to_numeric` or `df.to_datetime`, because Without coerce → ❌ conversion error because `"unknown"` isn't a number. when we use this to not get that error. There are also other options we can use here:
```python
errors='raise'   # default → throw an error
errors='coerce'  # invalid → NaN
errors='ignore'  # leave the original value
```